# تدريب المُشكِّل بالحجم الكبير
**قبل البدء:** وقت التشغيل ← تغيير نوع وقت التشغيل ← **T4 GPU** ← حفظ.
شغّل الخلايا **واحدة واحدة بالترتيب**، وانتظر حتى تظهر علامة ✓ خضراء قبل أن تشغّل التي بعدها. كل خلية تطبع في آخرها `تم ✔` عند النجاح؛ إن لم تجدها فأرسل لي لقطة لمخرجات الخلية.

In [ ]:
%%bash
set -e
cd /content
rm -rf dhad Open-Hadith-Data arabic-text-diacritization
git clone -q --depth 1 https://github.com/kaledgmal71-commits/dhad.git
git clone -q --depth 1 https://github.com/mhashim6/Open-Hadith-Data.git
git clone -q --depth 1 https://github.com/AliOsm/arabic-text-diacritization.git
test -f dhad/tools/dia_train.py && test -f dhad/tools/dia_eval.py && test -f arabic-text-diacritization/dataset/test.txt
nvidia-smi -L || echo "تنبيه: لا توجد بطاقة رسوم، فعّل T4 GPU من نوع وقت التشغيل"
echo "تم ✔ التنزيل"

In [ ]:
%%bash
set -e
python3 -c "import cupy; print('cupy', cupy.__version__)" 2>/dev/null || pip -q install cupy-cuda12x
python3 -c "import cupy; print('cupy', cupy.__version__, 'GPU OK' if cupy.cuda.runtime.getDeviceCount()>0 else 'NO GPU')"
echo "تم ✔ المكتبة"

In [ ]:
%%bash
set -e
cd /content
python3 dhad/tools/diacritizer_data.py Open-Hadith-Data hadith.npz
test -f hadith.npz
echo "تم ✔ البيانات"

In [ ]:
%%bash
set -e
cd /content
python3 dhad/tools/dia_train.py hadith.npz smoke --minutes 2 --d 384 --layers 6 --heads 6 --ff 1536 --T 256 --batch 64 --val arabic-text-diacritization/dataset/val.txt
test -f smoke.npz
echo "تم ✔ الاختبار السريع (انظر سطر params وسرعة step)"

In [ ]:
%%bash
set -e
cd /content
python3 dhad/tools/dia_train.py hadith.npz dia_big --minutes 120 --d 384 --layers 6 --heads 6 --ff 1536 --T 256 --batch 64 --lr 5e-4 --val arabic-text-diacritization/dataset/val.txt
test -f dia_big.npz
echo "تم ✔ التدريب"

In [ ]:
%%bash
set -e
cd /content
python3 dhad/tools/dia_eval.py dia_big arabic-text-diacritization/dataset/test.txt
echo "تم ✔ القياس (مرة واحدة فقط)"

In [ ]:
%%bash
cd /content
zip -q dia_big.zip dia_big.npz dia_big.json dia_big.log && echo "تم ✔ الضغط"

In [ ]:
from google.colab import files
files.download('/content/dia_big.zip')